# ICU Discharge Risk & Intervention Bot (READMIN)
## MS Health Informatics — Hofstra University | MIMIC-IV Demo v2.2
**Goal:** Build a risk score (0–100) that predicts 30/60/90-day ICU readmission at discharge,
and assign each patient a Low / Medium / High intervention tier.

---
**Run every cell top to bottom. All outputs are saved to the `outputs/` folder.**

## SECTION 0 — Install & Import Libraries

In [1]:
# Run this cell first — installs everything needed
# In Google Colab: this is all you need
# In Jupyter locally: run 'pip install ...' in your terminal first

# !pip install pandas numpy scikit-learn xgboost shap joblib matplotlib seaborn
!pip install imbalanced-learn --quiet  # for SMOTE
!pip install comorbidipy --quiet  # Elixhauser score

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import shap
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# ── Output folders ──────────────────────────────────────────────────────────
BASE = '/content/drive/MyDrive/IDRIB_project/'
for folder in [BASE+'data/processed', BASE+'models', BASE+'outputs', BASE+'outputs/charts']:
    os.makedirs(folder, exist_ok=True)

print('✅ All libraries imported successfully.')
print('✅ Output folders ready.')

✅ All libraries imported successfully.
✅ Output folders ready.


---
## SECTION 1 — Data Loading
Load the 7 MIMIC-IV tables we need. All filtered to the 100 demo patients.

In [2]:
import os, gzip, shutil, pandas as pd

# ── Create all missing folders ────────────────────────────────────────────
folders = [
    BASE + 'data/raw',
    BASE + 'data/processed',
    BASE + 'models',
    BASE + 'outputs/charts',
    BASE + 'notebooks',
]
for f in folders:
    os.makedirs(f, exist_ok=True)
    print(f'✅ {f}')

print('\nAll folders ready.')

✅ /content/drive/MyDrive/IDRIB_project/data/raw
✅ /content/drive/MyDrive/IDRIB_project/data/processed
✅ /content/drive/MyDrive/IDRIB_project/models
✅ /content/drive/MyDrive/IDRIB_project/outputs/charts
✅ /content/drive/MyDrive/IDRIB_project/notebooks

All folders ready.


In [3]:
print("Files in data/raw:")
for f in os.listdir(BASE + 'data/raw'):
    print(f'  {f}')

Files in data/raw:


In [4]:
import pandas as pd, os

DATA_PATH = '/content/drive/MyDrive/IDRIB_project/data/raw/'

demo_ids_list = [
    10000032,10001217,10001725,10002428,10002495,10002930,10003046,
    10003400,10004235,10005011,10005515,10005765,10006008,10006580,
    10007085,10007795,10008183,10009628,10010116,10010717,10011883,
    10012853,10013310,10013502,10014354,10014729,10015496,10015628,
    10016150,10016742,10017025,10017045,10018044,10018328,10018568,
    10019003,10019385,10020306,10020740,10021214,10021492,10021888,
    10022218,10022430,10022641,10023499,10023541,10024051,10024108,
    10024600,10025174,10025297,10025756,10026006,10026299,10026354,
    10026597,10027103,10027602,10028134,10028341,10029327,10030050,
    10030273,10030743,10031765,10032441,10033094,10033667,10034361,
    10035631,10036041,10036267,10036838,10037052,10037417,10038034,
    10038332,10038878,10039367,10039708,10040005,10040310,10040666,
    10041733,10042018,10042101,10042281,10042516,10043137,10043346,
    10043881,10044058,10044083,10044307,10045084,10046166,10046416,
    10046471,10046499
]

save_path = DATA_PATH + 'demo_subject_id.csv'
pd.DataFrame({'subject_id': demo_ids_list}).to_csv(save_path, index=False)

# Verify it actually exists on Drive now
if os.path.exists(save_path):
    print(f'✅ Saved to Drive: {save_path}')
    print(f'   File size: {os.path.getsize(save_path)} bytes')
else:
    print('❌ Still not found — Drive may not be syncing properly')

✅ Saved to Drive: /content/drive/MyDrive/IDRIB_project/data/raw/demo_subject_id.csv
   File size: 911 bytes


In [5]:
import os, gzip, shutil, subprocess

# Force Drive to sync and show files
subprocess.run(['ls', '-la', BASE], capture_output=False)

CompletedProcess(args=['ls', '-la', '/content/drive/MyDrive/IDRIB_project/'], returncode=0)

In [7]:
import os, subprocess
from google.colab import drive

subprocess.run(['rm', '-rf', '/content/drive'], check=False)
drive.mount('/content/drive')

print("Files in raw/:")
for f in sorted(os.listdir(BASE)):
    print(f'  {f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in raw/:


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IDRIB_project/'

In [ ]:
import os
from google.colab import drive

# Full remount
drive.flush_and_unmount()
drive.mount('/content/drive')

# Now check
BASE = '/content/drive/MyDrive/IDRIB_project/data/raw/'
print("Files in raw/:")
for f in os.listdir(BASE):
    print(f'  {f}')

In [ ]:
import os, gzip, shutil, pandas as pd

BASE = '/content/drive/MyDrive/IDRIB_project/data/raw/'

needed = [
    'admissions.csv.gz',
    'patients.csv.gz',
    'icustays.csv.gz',
    'diagnoses_icd.csv.gz',
    'labevents.csv.gz',
    'prescriptions.csv.gz',
    'chartevents.csv.gz',
]

print('Unzipping files into Drive...')
for fname in needed:
    gz  = BASE + fname
    csv = BASE + fname.replace('.gz', '')
    if os.path.exists(csv):
        print(f'  already done: {fname}')
        continue
    print(f'  unzipping {fname} ...', end=' ', flush=True)
    with gzip.open(gz, 'rb') as f_in, open(csv, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print('✅')

print('\nAll done! CSV files in raw/:')
for f in sorted(os.listdir(BASE)):
    if f.endswith('.csv'):
        mb = os.path.getsize(BASE + f) / 1024 / 1024
        print(f'  {f:40s} {mb:.1f} MB')

In [ ]:
import os, shutil, gzip, pandas as pd
from google.colab import drive

# ── Mount Drive (safe — checks if already mounted) ────────────────────────────
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted ✅')

DATA_PATH = '/content/drive/MyDrive/IDRIB_project/data/raw/'

# ── Unzip any files not yet unzipped ─────────────────────────────────────────
needed = ['admissions.csv','patients.csv','icustays.csv',
          'diagnoses_icd.csv','labevents.csv','prescriptions.csv','chartevents.csv']
for fname in needed:
    csv_path = DATA_PATH + fname
    gz_path  = csv_path + '.gz'
    if not os.path.exists(csv_path) and os.path.exists(gz_path):
        print(f'Unzipping {fname}...')
        with gzip.open(gz_path,'rb') as f_in, open(csv_path,'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        print(f'  done: {fname}')

# ── Load demo subject IDs ─────────────────────────────────────────────────────
demo_ids = pd.read_csv(DATA_PATH + 'demo_subject_id.csv')['subject_id'].tolist()
print(f'Demo patients: {len(demo_ids)} subject IDs loaded')

# ── Helper: load and filter to demo patients ─────────────────────────────────
def load(filename, id_col='subject_id'):
    df = pd.read_csv(DATA_PATH + filename)
    df = df[df[id_col].isin(demo_ids)].copy()
    df.columns = df.columns.str.lower().str.strip()
    print(f'  {filename:30s}  →  {len(df):,} rows | {df.shape[1]} cols')
    return df

print('\nLoading MIMIC-IV tables...')
admissions    = load('admissions.csv')
patients      = load('patients.csv')
icustays      = load('icustays.csv')
diagnoses     = load('diagnoses_icd.csv')
labevents     = load('labevents.csv')
prescriptions = load('prescriptions.csv')
chartevents   = load('chartevents.csv')

print('\n✅ All tables loaded successfully!')

In [ ]:
# ── Quick look at each table ─────────────────────────────────────────────────
print('=== ADMISSIONS — first 3 rows ===')
display(admissions.head(3))

print('\n=== ICU STAYS — first 3 rows ===')
display(icustays.head(3))

print('\n=== PATIENTS — first 3 rows ===')
display(patients.head(3))

---
## SECTION 2 — Label Creation
For each ICU discharge, did the patient come back to the ICU within 30, 60, or 90 days?
This is the **target variable** the model will learn to predict.

In [ ]:
# ── Parse dates ──────────────────────────────────────────────────────────────
icustays['intime']  = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
admissions['admittime']    = pd.to_datetime(admissions['admittime'])
admissions['dischtime']    = pd.to_datetime(admissions['dischtime'])
admissions['deathtime']    = pd.to_datetime(admissions['deathtime'])

# ── ICU length of stay in hours ──────────────────────────────────────────────
icustays['icu_los_hours'] = (
    (icustays['outtime'] - icustays['intime']).dt.total_seconds() / 3600
)

# ── For each ICU stay find if there is a LATER ICU stay for the same patient ─
# Sort by patient and time
icu_sorted = icustays.sort_values(['subject_id', 'intime']).copy()

# Self-join: pair each stay with all later stays for the same patient
icu_next = icu_sorted[['subject_id', 'stay_id', 'intime', 'outtime', 'icu_los_hours']].copy()
icu_next.columns = ['subject_id', 'stay_id', 'intime', 'outtime', 'icu_los_hours']

merged = icu_next.merge(
    icu_next[['subject_id', 'intime']].rename(columns={'intime': 'next_intime'}),
    on='subject_id',
    how='left'
)

# Keep only rows where the next_intime is AFTER the current outtime (true readmission)
merged = merged[merged['next_intime'] > merged['outtime']]

# For each stay, find the EARLIEST next admission (minimum gap)
earliest = (
    merged.groupby('stay_id')['next_intime'].min().reset_index()
    .rename(columns={'next_intime': 'first_readmit_time'})
)

# Merge back
labels = icu_next.merge(earliest, on='stay_id', how='left')

# ── Compute days to readmission ───────────────────────────────────────────────
labels['days_to_readmit'] = (
    (labels['first_readmit_time'] - labels['outtime']).dt.total_seconds() / 86400
)

# ── Binary labels for each window ────────────────────────────────────────────
labels['readmit_30d'] = (labels['days_to_readmit'] <= 30).astype(int)
labels['readmit_60d'] = (labels['days_to_readmit'] <= 60).astype(int)
labels['readmit_90d'] = (labels['days_to_readmit'] <= 90).astype(int)

# Fill NaN (no readmission found) → 0
for col in ['readmit_30d', 'readmit_60d', 'readmit_90d']:
    labels[col] = labels[col].fillna(0).astype(int)

print('=== Readmission Rates ===')
for col, label in [('readmit_30d', '30-day'), ('readmit_60d', '60-day'), ('readmit_90d', '90-day')]:
    n = labels[col].sum()
    pct = n / len(labels) * 100
    print(f'  {label}: {n} readmissions ({pct:.1f}%)')

print(f'\n  Total ICU stays in dataset: {len(labels)}')
print('\n✅ Labels created.')

In [ ]:
# ── Visualize readmission distribution ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ['#378ADD', '#1D9E75', '#BA7517']

for ax, col, label, color in zip(
    axes,
    ['readmit_30d', 'readmit_60d', 'readmit_90d'],
    ['30-Day', '60-Day', '90-Day'],
    colors
):
    counts = labels[col].value_counts().sort_index()
    ax.bar(['No Readmit', 'Readmit'], counts.values, color=['#D3D1C7', color], width=0.5)
    ax.set_title(f'{label} Readmission', fontsize=13, fontweight='bold')
    ax.set_ylabel('Patient Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.3, str(v), ha='center', fontsize=11)

plt.suptitle('ICU Readmission Label Distribution (MIMIC-IV Demo)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/01_readmission_distribution.png', bbox_inches='tight')
plt.show()
print('Chart saved → /content/drive/MyDrive/IDRIB_project/outputs/charts/01_readmission_distribution.png')

---
## SECTION 3 — Feature Engineering
Extract all clinical variables that predict readmission risk.
Each row = one ICU stay. Each column = one feature.

In [ ]:
# ── FEATURE 1: Demographics ──────────────────────────────────────────────────
print('Building demographic features...')

# Anchor year from patients table to estimate age
pat = patients[['subject_id', 'anchor_age', 'anchor_year', 'gender']].copy()
pat['gender_male'] = (pat['gender'] == 'M').astype(int)

# Map insurance and marital status from admissions
adm_demo = admissions[[
    'subject_id', 'hadm_id', 'insurance', 'marital_status',
    'race', 'discharge_location', 'hospital_expire_flag'
]].drop_duplicates(subset=['hadm_id'])

# Join icustays → hadm_id → demographics
feat = labels[['stay_id', 'subject_id', 'intime', 'outtime',
               'icu_los_hours', 'readmit_30d', 'readmit_60d', 'readmit_90d']].copy()

feat = feat.merge(icustays[['stay_id', 'hadm_id', 'first_careunit']], on='stay_id', how='left')
feat = feat.merge(pat[['subject_id', 'anchor_age', 'gender_male']], on='subject_id', how='left')
feat = feat.merge(adm_demo, on=['subject_id', 'hadm_id'], how='left')

# Encode discharge location (home vs other)
feat['discharge_home'] = feat['discharge_location'].str.lower().str.contains(
    'home', na=False).astype(int)

# Encode insurance (Medicare = higher risk)
feat['insurance_medicare'] = feat['insurance'].str.lower().str.contains(
    'medicare', na=False).astype(int)

# ICU LOS in days
feat['icu_los_days'] = feat['icu_los_hours'] / 24

print(f'  Demographics merged. Shape: {feat.shape}')

In [ ]:
# ── FEATURE 2: Prior ICU admissions count ─────────────────────────────────────
print('Building prior admissions feature...')

prior_icu = []
for _, row in feat.iterrows():
    n_prior = (
        (icustays['subject_id'] == row['subject_id']) &
        (icustays['intime'] < row['intime'])
    ).sum()
    prior_icu.append(n_prior)

feat['n_prior_icu'] = prior_icu
print(f'  Prior ICU admissions: max={feat["n_prior_icu"].max()}, mean={feat["n_prior_icu"].mean():.1f}')

In [ ]:
# ── FEATURE 3: Comorbidity count from ICD diagnosis codes ────────────────────
print('Building comorbidity features...')

# Count unique ICD codes per hospitalization as a proxy for comorbidity burden
dx_count = (
    diagnoses.groupby('hadm_id')['icd_code']
    .nunique()
    .reset_index()
    .rename(columns={'icd_code': 'n_icd_codes'})
)

# Flag key comorbidities using ICD-10 prefixes
# These are the highest-weight readmission predictors
def flag_comorbidity(hadm_ids, prefix):
    """Return 1 if any ICD code for that admission starts with prefix."""
    hits = diagnoses[
        diagnoses['icd_code'].str.startswith(prefix, na=False)
    ]['hadm_id'].unique()
    return hadm_ids.isin(hits).astype(int)

dx_flags = diagnoses[['hadm_id']].drop_duplicates().copy()
dx_flags['has_chf']       = flag_comorbidity(dx_flags['hadm_id'], 'I50')   # Heart failure
dx_flags['has_ckd']       = flag_comorbidity(dx_flags['hadm_id'], 'N18')   # Chronic kidney disease
dx_flags['has_copd']      = flag_comorbidity(dx_flags['hadm_id'], 'J44')   # COPD
dx_flags['has_diabetes']  = flag_comorbidity(dx_flags['hadm_id'], 'E11')   # Type 2 diabetes
dx_flags['has_sepsis']    = flag_comorbidity(dx_flags['hadm_id'], 'A41')   # Sepsis
dx_flags['has_pneumonia'] = flag_comorbidity(dx_flags['hadm_id'], 'J18')   # Pneumonia

dx_flags = dx_flags.merge(dx_count, on='hadm_id', how='left')

feat = feat.merge(dx_flags, on='hadm_id', how='left')

# Total comorbidity score = sum of flags
comorbidity_cols = ['has_chf', 'has_ckd', 'has_copd', 'has_diabetes', 'has_sepsis', 'has_pneumonia']
feat['comorbidity_score'] = feat[comorbidity_cols].fillna(0).sum(axis=1)

print(f'  Comorbidity features added. Columns added: {len(comorbidity_cols) + 2}')

### Feature 3b — Elixhauser Comorbidity Score + Ventilation & Vasopressors
Elixhauser is the validated clinical comorbidity index named in the proposal. Ventilation and vasopressor use are strong readmission predictors.

In [ ]:
# ── FEATURE 3b: Elixhauser Comorbidity Score ─────────────────────────────────
print('Building Elixhauser comorbidity score...')

# Manual Elixhauser — 6 key condition groups mapped to ICD-10 prefixes
ELIX_CONDITIONS = {
    'elix_chf'         : ['I099','I110','I130','I132','I255','I420',
                           'I425','I426','I427','I428','I429','I43','I50','P290'],
    'elix_arrhythmia'  : ['I441','I442','I443','I456','I459','I47','I48',
                           'I49','R000','R001','R008','T821','Z450','Z950'],
    'elix_renal'       : ['I120','I131','N18','N19','N250',
                           'Z490','Z491','Z492','Z940','Z992'],
    'elix_liver'       : ['B18','I85','I864','I982','K70','K711','K713',
                           'K714','K715','K717','K72','K73','K74',
                           'K760','K762','K763','K764','K768','K769','Z944'],
    'elix_cancer'      : ['C0','C1','C2','C3','C4','C5','C6',
                           'C70','C71','C72','C73','C74','C75',
                           'C76','C81','C82','C83','C84','C85','C88','C9'],
    'elix_depression'  : ['F204','F313','F314','F315','F32',
                           'F33','F341','F412','F432'],
}

# Build per-admission flags
elix_rows = []
for hid in feat['hadm_id'].unique():
    codes = diagnoses[diagnoses['hadm_id']==hid]['icd_code'].tolist()
    row = {'hadm_id': hid}
    score = 0
    for cond, prefixes in ELIX_CONDITIONS.items():
        flag = int(any(any(c.startswith(p) for p in prefixes) for c in codes))
        row[cond] = flag
        score += flag
    row['elixhauser_score'] = score
    elix_rows.append(row)

elix_df = pd.DataFrame(elix_rows)
feat = feat.merge(elix_df, on='hadm_id', how='left')
print(f'  Elixhauser score: mean={feat["elixhauser_score"].mean():.1f}, '
      f'max={feat["elixhauser_score"].max():.0f}')

# ── FEATURE 3c: Ventilation & Vasopressor flags ────────────────────────────────
print('\nBuilding ventilation & vasopressor features...')

# Ventilation itemids (invasive mechanical ventilation in chartevents)
VENT_ITEMS = [720, 467, 468, 469, 470, 471, 227287,
              224687, 224685, 224684, 224686, 224696, 224695, 224694]

# Vasopressor drug names in prescriptions
VASO_DRUGS = ['norepinephrine','epinephrine','dopamine',
              'vasopressin','phenylephrine','dobutamine']

vent_flags, vaso_flags = [], []
for _, row in feat.iterrows():
    # Ventilation
    if 'itemid' in chartevents.columns:
        vent = chartevents[
            (chartevents['subject_id']==row['subject_id']) &
            (chartevents['itemid'].isin(VENT_ITEMS))
        ]
        vent_flags.append(1 if len(vent)>0 else 0)
    else:
        vent_flags.append(0)
    # Vasopressor
    if 'drug' in prescriptions.columns:
        pat_drugs = prescriptions[
            prescriptions['hadm_id']==row['hadm_id']
        ]['drug'].str.lower().fillna('').tolist()
        vaso = any(any(v in d for v in VASO_DRUGS) for d in pat_drugs)
        vaso_flags.append(1 if vaso else 0)
    else:
        vaso_flags.append(0)

feat['ventilation_flag'] = vent_flags
feat['vasopressor_flag'] = vaso_flags

print(f'  Ventilation: {sum(vent_flags)} patients ({sum(vent_flags)/len(vent_flags):.1%})')
print(f'  Vasopressor: {sum(vaso_flags)} patients ({sum(vaso_flags)/len(vaso_flags):.1%})')
print('✅ Elixhauser + Ventilation + Vasopressor done.')


In [ ]:
# ── FEATURE 4: Lab values at discharge ───────────────────────────────────────
# We take the LAST recorded value before ICU discharge for each key lab
print('Building lab features...')

# Key MIMIC-IV itemids for common labs
LAB_ITEMS = {
    'creatinine' : 50912,   # Creatinine (renal function)
    'wbc'        : 51301,   # White blood cells (infection/inflammation)
    'lactate'    : 50813,   # Lactate (tissue perfusion)
    'hemoglobin' : 51222,   # Hemoglobin (anemia)
    'sodium'     : 50983,   # Sodium (electrolyte balance)
    'bun'        : 51006,   # BUN (kidney function)
}

if 'itemid' in labevents.columns and 'valuenum' in labevents.columns:
    labevents['charttime'] = pd.to_datetime(labevents['charttime'])

    lab_features_list = []
    for _, row in feat.iterrows():
        # Get all labs for this patient before discharge time
        pat_labs = labevents[
            (labevents['subject_id'] == row['subject_id']) &
            (labevents['charttime'] <= row['outtime'])
        ].copy()

        row_labs = {'stay_id': row['stay_id']}
        for lab_name, itemid in LAB_ITEMS.items():
            lab_vals = pat_labs[
                pat_labs['itemid'] == itemid
            ].sort_values('charttime')
            if len(lab_vals) > 0:
                row_labs[f'lab_{lab_name}'] = lab_vals['valuenum'].iloc[-1]  # last value
            else:
                row_labs[f'lab_{lab_name}'] = np.nan
        lab_features_list.append(row_labs)

    lab_df = pd.DataFrame(lab_features_list)
    feat = feat.merge(lab_df, on='stay_id', how='left')
    print(f'  Lab features added: {list(LAB_ITEMS.keys())}')
else:
    print('  labevents structure differs — adding NaN placeholders for labs')
    for lab_name in LAB_ITEMS:
        feat[f'lab_{lab_name}'] = np.nan

print('  ✅ Lab features done.')

In [ ]:
# ── FEATURE 5: Vitals in last 24hrs of ICU stay ───────────────────────────────
print('Building vitals features...')

# Key chartevents itemids for vitals
VITAL_ITEMS = {
    'heart_rate'  : 220045,
    'sbp'         : 220179,   # Systolic BP
    'dbp'         : 220180,   # Diastolic BP
    'o2_sat'      : 220277,   # SpO2
    'resp_rate'   : 220210,
    'temperature' : 223761,
}

if 'itemid' in chartevents.columns and 'valuenum' in chartevents.columns:
    chartevents['charttime'] = pd.to_datetime(chartevents['charttime'])

    vital_features_list = []
    for _, row in feat.iterrows():
        window_start = row['outtime'] - pd.Timedelta(hours=24)
        pat_vitals = chartevents[
            (chartevents['subject_id'] == row['subject_id']) &
            (chartevents['charttime'] >= window_start) &
            (chartevents['charttime'] <= row['outtime'])
        ].copy()

        row_vitals = {'stay_id': row['stay_id']}
        for v_name, itemid in VITAL_ITEMS.items():
            v_vals = pat_vitals[pat_vitals['itemid'] == itemid]['valuenum']
            if len(v_vals) > 0:
                row_vitals[f'vital_{v_name}_mean'] = v_vals.mean()
                row_vitals[f'vital_{v_name}_min']  = v_vals.min()
                row_vitals[f'vital_{v_name}_max']  = v_vals.max()
            else:
                for suffix in ['mean', 'min', 'max']:
                    row_vitals[f'vital_{v_name}_{suffix}'] = np.nan
        vital_features_list.append(row_vitals)

    vital_df = pd.DataFrame(vital_features_list)
    feat = feat.merge(vital_df, on='stay_id', how='left')
    print(f'  Vitals features added ({len(VITAL_ITEMS) * 3} columns)')
else:
    print('  chartevents structure differs — adding NaN placeholders for vitals')
    for v_name in VITAL_ITEMS:
        for suffix in ['mean', 'min', 'max']:
            feat[f'vital_{v_name}_{suffix}'] = np.nan

print('  ✅ Vitals features done.')

### Feature 5b — Time-Series Vitals (Trend Features)
Instead of just the last 24hrs, we capture how vitals are *trending* over the full ICU stay.

In [ ]:
# ── FEATURE 5b: Time-series vital trends over full ICU stay ──────────────────
# Key insight: is creatinine RISING or FALLING? Trajectory matters more than level.
print('Building time-series trend features...')

TREND_VITALS = {
    'heart_rate' : 220045,
    'sbp'        : 220179,
    'o2_sat'     : 220277,
    'resp_rate'  : 220210,
}

TREND_LABS = {
    'creatinine' : 50912,
    'wbc'        : 51301,
    'lactate'    : 50813,
}

def compute_trend(values):
    """Slope of linear regression through time-series values.
    Positive = worsening trend, Negative = improving trend."""
    if len(values) < 2:
        return 0.0
    x = np.arange(len(values))
    slope = np.polyfit(x, values, 1)[0]
    return slope

ts_features_list = []

if 'itemid' in chartevents.columns and 'valuenum' in chartevents.columns:
    for _, row in feat.iterrows():
        row_ts = {'stay_id': row['stay_id']}

        # Vital trends — full ICU stay window
        pat_chart = chartevents[
            (chartevents['subject_id'] == row['subject_id']) &
            (chartevents['charttime'] >= row['intime']) &
            (chartevents['charttime'] <= row['outtime'])
        ].sort_values('charttime')

        for v_name, itemid in TREND_VITALS.items():
            vals = pat_chart[pat_chart['itemid']==itemid]['valuenum'].dropna().values
            row_ts[f'trend_{v_name}'] = compute_trend(vals)
            row_ts[f'std_{v_name}']   = float(np.std(vals)) if len(vals)>1 else 0.0

        # Lab trends — full stay
        pat_labs = labevents[
            (labevents['subject_id'] == row['subject_id']) &
            (labevents['charttime'] >= row['intime']) &
            (labevents['charttime'] <= row['outtime'])
        ].sort_values('charttime') if 'charttime' in labevents.columns else pd.DataFrame()

        for l_name, itemid in TREND_LABS.items():
            if len(pat_labs) > 0:
                vals = pat_labs[pat_labs['itemid']==itemid]['valuenum'].dropna().values
            else:
                vals = np.array([])
            row_ts[f'trend_{l_name}'] = compute_trend(vals)

        ts_features_list.append(row_ts)

    ts_df = pd.DataFrame(ts_features_list)
    feat = feat.merge(ts_df, on='stay_id', how='left')

    ts_cols = [c for c in feat.columns if c.startswith('trend_') or c.startswith('std_')]
    print(f'  Time-series features added: {ts_cols}')
else:
    print('  chartevents not available — skipping time-series features')
    ts_cols = []

print('\n✅ Time-series features done.')


In [ ]:
# ── FEATURE 6: Medication count at discharge ──────────────────────────────────
print('Building medication features...')

if 'hadm_id' in prescriptions.columns:
    med_count = (
        prescriptions.groupby('hadm_id')['drug']
        .nunique()
        .reset_index()
        .rename(columns={'drug': 'n_medications'})
    )
    feat = feat.merge(med_count, on='hadm_id', how='left')
    print(f'  Medication count: mean={feat["n_medications"].mean():.1f}')
else:
    feat['n_medications'] = np.nan
    print('  prescriptions structure differs — NaN placeholder added')

print('\n✅ All features built.')
print(f'   Feature matrix shape: {feat.shape}')

---
## SECTION 4 — Preprocessing
Clean the feature matrix: handle missing values, select model features, save to CSV.

In [ ]:
# ── Define model feature columns (v4 — all features) ────────────────────────
FEATURE_COLS = [
    # Demographics
    'anchor_age', 'gender_male', 'insurance_medicare', 'discharge_home',
    # ICU stay
    'icu_los_days', 'n_prior_icu',
    # Comorbidities — custom flags + Elixhauser
    'comorbidity_score', 'n_icd_codes', 'elixhauser_score',
    'has_chf', 'has_ckd', 'has_copd', 'has_diabetes', 'has_sepsis', 'has_pneumonia',
    # Ventilation & vasopressors (NEW)
    'ventilation_flag', 'vasopressor_flag',
    # Labs
    'lab_creatinine', 'lab_wbc', 'lab_lactate',
    'lab_hemoglobin', 'lab_sodium', 'lab_bun',
    # Vitals last 24hrs
    'vital_heart_rate_mean', 'vital_sbp_mean',
    'vital_o2_sat_min', 'vital_resp_rate_mean',
    # Time-series trends
    'trend_heart_rate', 'trend_sbp', 'trend_o2_sat', 'trend_resp_rate',
    'std_heart_rate', 'std_sbp', 'std_o2_sat', 'std_resp_rate',
    'trend_creatinine', 'trend_wbc', 'trend_lactate',
    # Medications
    'n_medications',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in feat.columns]
print(f'Total features: {len(FEATURE_COLS)}')
new_feats = [c for c in ['elixhauser_score','ventilation_flag','vasopressor_flag'] if c in FEATURE_COLS]
print(f'New features added: {new_feats}')

TARGET = 'readmit_30d'
model_df = feat[['stay_id','subject_id','outtime'] + FEATURE_COLS +
                ['readmit_30d','readmit_60d','readmit_90d']].copy()
missing = model_df[FEATURE_COLS].isnull().sum()
missing_pct = (missing/len(model_df)*100).round(1)
missing_df = pd.DataFrame({'n':missing,'%':missing_pct})
missing_df = missing_df[missing_df['n']>0].sort_values('%',ascending=False)
print('\nMissing values:')
print(missing_df.to_string())


In [ ]:
 # ── Impute missing values with column median ──────────────────────────────────
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(
    imputer.fit_transform(model_df[FEATURE_COLS]),
    columns=FEATURE_COLS
)
y = model_df[TARGET].values

print(f'X shape: {X.shape}  (rows=ICU stays, cols=features)')
print(f'y shape: {y.shape}  (target: {TARGET})')
print(f'Positive class rate: {y.mean():.2%} readmitted')

# ── Save clean feature matrix ─────────────────────────────────────────────────
feature_matrix = pd.concat([
    model_df[['stay_id', 'subject_id', 'outtime', 'readmit_30d', 'readmit_60d', 'readmit_90d']].reset_index(drop=True),
    X
], axis=1)

feature_matrix.to_csv('/content/drive/MyDrive/IDRIB_project/data/processed/features_matrix.csv', index=False)
print('\n✅ features_matrix.csv saved → data/processed/')

### SMOTE — Synthetic Minority Oversampling
Creates synthetic readmission examples to fix the 85/15 class imbalance.

In [ ]:
# ── Step 1: Train/Test Split FIRST — then SMOTE only on training data ─────────
# RULE: Split the data BEFORE any resampling.
# The test set must NEVER be touched by SMOTE — it must stay real patients only.
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter

# 80/20 stratified split — preserves 15% readmission rate in both sets
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {len(X_train_raw)} patients | '
      f'Readmit rate: {y_train_raw.mean():.1%}')
print(f'Test set:  {len(X_test)} patients  | '
      f'Readmit rate: {y_test.mean():.1%}')
print(f'Test set stays REAL — no SMOTE ever applied to it.')

# Step 2: Apply SMOTE ONLY to training data
smote = SMOTE(random_state=42, k_neighbors=min(4, y_train_raw.sum()-1))
try:
    X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)
    print(f'\nBefore SMOTE (train): {Counter(y_train_raw)}')
    print(f'After SMOTE  (train): {Counter(y_train)}')
    print(f'Training size grew: {len(y_train_raw)} → {len(y_train)} rows')
    USE_SMOTE = True
except Exception as e:
    print(f'SMOTE skipped: {e}')
    X_train, y_train = X_train_raw.copy(), y_train_raw.copy()
    USE_SMOTE = False

# Visualize before/after
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, counts, title in zip(axes,
    [Counter(y), Counter(y_train_raw), Counter(y_train)],
    ['Full Dataset\n(Before split)', 'Training Set\n(Before SMOTE)', 'Training Set\n(After SMOTE)']):
    ax.bar(['No Readmit','Readmit'], [counts[0], counts[1]],
           color=['#1D9E75','#E24B4A'], alpha=0.85, width=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Count')
    for j, v in enumerate([counts[0], counts[1]]):
        ax.text(j, v+0.5, str(v), ha='center', fontsize=12, fontweight='bold')

plt.suptitle('Correct SMOTE Workflow: Split First → SMOTE Only on Training Data',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/02b_smote_correct.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n✅ Train/test split done. SMOTE applied to training set only.')
print(f'Test set: {len(X_test)} real patients held out for final evaluation.')


In [ ]:
# ── Feature distribution plots ────────────────────────────────────────────────
key_features = ['anchor_age', 'icu_los_days', 'comorbidity_score', 'n_prior_icu',
                'lab_creatinine', 'lab_wbc', 'n_medications']
key_features = [f for f in key_features if f in X.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]
    readmit = X[col][y == 1]
    no_readmit = X[col][y == 0]
    ax.hist(no_readmit, bins=15, alpha=0.6, label='No Readmit', color='#D3D1C7')
    ax.hist(readmit,    bins=15, alpha=0.7, label='Readmit',    color='#E24B4A')
    ax.set_title(col.replace('_', ' ').title(), fontsize=11)
    ax.legend(fontsize=8)

for j in range(len(key_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Readmitted vs. Not Readmitted (30-Day)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/02_feature_distributions.png', bbox_inches='tight')
plt.show()
print('Chart saved → /content/drive/MyDrive/IDRIB_project/outputs/charts/02_feature_distributions.png')

---
## SECTION 5 — Model Training
Train three models. Use 5-fold cross-validation to evaluate each fairly.

In [ ]:
# ── Step 3: Models — CV on SMOTE-balanced training set + imblearn Pipeline ────
# Two layers of protection against leakage:
#   Layer 1: Train/test split already done — test set never seen here
#   Layer 2: imblearn Pipeline applies SMOTE inside each CV fold on training data
#            so synthetic patients never bleed into validation folds either

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import average_precision_score

k_nn = min(4, y_train_raw.sum() - 1)  # safe for small train set

MODELS = {
    'Logistic Regression': ImbPipeline([
        ('smote',  SMOTE(random_state=42, k_neighbors=k_nn)),
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=2000, random_state=42, C=0.1))
    ]),
    'Random Forest': ImbPipeline([
        ('smote', SMOTE(random_state=42, k_neighbors=k_nn)),
        ('clf',   RandomForestClassifier(n_estimators=300, max_depth=4,
                      random_state=42, min_samples_leaf=5, max_features='sqrt'))
    ]),
    'XGBoost': ImbPipeline([
        ('smote', SMOTE(random_state=42, k_neighbors=k_nn)),
        ('clf',   XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8, scale_pos_weight=4,
                      eval_metric='logloss', random_state=42, verbosity=0))
    ]),
    'Gradient Boosting': ImbPipeline([
        ('smote', SMOTE(random_state=42, k_neighbors=k_nn)),
        ('clf',   GradientBoostingClassifier(n_estimators=100, max_depth=2,
                      learning_rate=0.05, random_state=42,
                      subsample=0.8, min_samples_leaf=5))
    ]),
}

# CV runs on X_train_raw (pre-SMOTE) — pipeline applies SMOTE inside each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_roc, results_pr = {}, {}
print('Running 5-fold CV on training set (SMOTE inside each fold)...\n')
print(f'  Training set: {len(X_train_raw)} real patients')
print(f'  Test set:     {len(X_test)} real patients (held out — not used here)\n')

for name, model in MODELS.items():
    roc = cross_val_score(model, X_train_raw, y_train_raw,
                          cv=cv, scoring='roc_auc', n_jobs=-1)
    pr  = cross_val_score(model, X_train_raw, y_train_raw,
                          cv=cv, scoring='average_precision', n_jobs=-1)
    results_roc[name] = roc
    results_pr[name]  = pr
    print(f'{name:25s}  ROC-AUC: {roc.mean():.3f} ± {roc.std():.3f}'
          f'  |  PR-AUC: {pr.mean():.3f} ± {pr.std():.3f}')

results = results_roc
baseline_pr = y_train_raw.mean()
print(f'\nBaseline PR-AUC (random): {baseline_pr:.3f}')
best_cv = max(results_roc, key=lambda m: results_roc[m].mean())
print(f'Best ROC-AUC: {best_cv} ({results_roc[best_cv].mean():.3f})')
best_pr = max(results_pr,  key=lambda m: results_pr[m].mean())
print(f'Best PR-AUC:  {best_pr}  ({results_pr[best_pr].mean():.3f})')
print('\n✅ Cross-validation complete.')


In [ ]:
# ── Model comparison chart — improved ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
model_names = list(results.keys())
means = [results[m].mean() for m in model_names]
stds  = [results[m].std()  for m in model_names]
colors_bar = ['#378ADD', '#1D9E75', '#BA7517', '#7F77DD']
bars = ax.barh(model_names, means, xerr=stds,
               color=colors_bar[:len(model_names)], alpha=0.85,
               height=0.5, capsize=6, error_kw={'linewidth':2,'color':'#444441'})
for bar, mean in zip(bars, means):
    ax.text(mean - 0.03, bar.get_y() + bar.get_height()/2,
            f'{mean:.3f}', va='center', fontsize=12, fontweight='bold', color='white')
ax.axvline(x=0.80, color='#E24B4A', linestyle='--', lw=2, label='Target AUROC=0.80 (full dataset)')
ax.axvline(x=0.50, color='gray',    linestyle=':',  lw=1.5, label='Random chance=0.50')
ax.set_xlabel('AUROC (5-fold Cross-Validation)', fontsize=13)
ax.set_title('Model Comparison — 30-Day ICU Readmission\n(Demo: 100 patients — AUROC improves with full 65K dataset)', fontsize=12, fontweight='bold')
ax.set_xlim(0.35, 1.02)
ax.legend(fontsize=10)
ax.tick_params(axis='y', labelsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/03_model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved.')


---
## SECTION 6 — Best Model Evaluation
Train the best model on all data. Plot ROC curve. Save the model.

In [ ]:
# ── Step 4: Final model — fit on SMOTE training set, evaluate on HELD-OUT test set
# This is the honest final evaluation — test patients were never seen during training
best_name = max(results_roc, key=lambda m: results_roc[m].mean())
print(f'Best model: {best_name}  (CV AUROC = {results_roc[best_name].mean():.3f})')

best_model = MODELS[best_name]
best_model.fit(X_train_raw, y_train_raw)  # fit on real train data (pipeline adds SMOTE)

# Evaluate on held-out test set — real patients only
y_prob_test = best_model.predict_proba(X_test)[:, 1]
test_roc = roc_auc_score(y_test, y_prob_test)
test_pr  = average_precision_score(y_test, y_prob_test)

# Also get probabilities for full dataset (for scoring all patients)
y_prob = best_model.predict_proba(X)[:, 1]

print(f'\n=== FINAL TEST SET EVALUATION (held-out — never seen during training) ===')
print(f'  Test ROC-AUC : {test_roc:.3f}')
print(f'  Test PR-AUC  : {test_pr:.3f}  (baseline = {y_test.mean():.3f})')
print(f'  Test patients: {len(X_test)} ({y_test.sum()} readmitted)')

# ROC curves on test set for all models
fig, ax = plt.subplots(figsize=(8, 7))
roc_colors = ['#378ADD','#1D9E75','#BA7517','#7F77DD']
for (name, model), color in zip(MODELS.items(), roc_colors):
    model.fit(X_train_raw, y_train_raw)
    yp   = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, yp)
    auc  = roc_auc_score(y_test, yp)
    lw   = 3 if name == best_name else 1.5
    ls   = '-' if name == best_name else '--'
    ax.plot(fpr, tpr, color=color, lw=lw, linestyle=ls,
            label=f'{name} (AUC={auc:.3f}){" ← best" if name==best_name else ""}')
ax.plot([0,1],[0,1],'k:',lw=1,label='Random chance')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — Evaluated on Held-Out Test Set\n'
             '(Train: SMOTE-balanced | Test: Real patients only)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/04_roc_test_set.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved → /content/drive/MyDrive/IDRIB_project/outputs/charts/04_roc_test_set.png')


In [ ]:
# ── Save model and feature list ───────────────────────────────────────────────
joblib.dump(best_model, '/content/drive/MyDrive/IDRIB_project/models/risk_model.pkl')
joblib.dump(FEATURE_COLS, '/content/drive/MyDrive/IDRIB_project/models/feature_cols.pkl')
joblib.dump(imputer, '/content/drive/MyDrive/IDRIB_project/models/imputer.pkl')

print(f'✅ Model saved → /content/drive/MyDrive/IDRIB_project/models/risk_model.pkl')
print(f'✅ Feature list saved → /content/drive/MyDrive/IDRIB_project/models/feature_cols.pkl')
print(f'✅ Imputer saved → /content/drive/MyDrive/IDRIB_project/models/imputer.pkl')

---
## SECTION 7 — Risk Scoring
Every ICU discharge gets a risk score 0–100 and a tier: Low / Medium / High.

In [ ]:
# ── Score all patients ────────────────────────────────────────────────────────
probs = best_model.predict_proba(X)[:, 1]
scores = (probs * 100).round(1)   # Scale 0–100

# ── Assign risk tier ──────────────────────────────────────────────────────────
def assign_tier(score):
    if score >= 70:
        return 'HIGH'
    elif score >= 40:
        return 'MEDIUM'
    else:
        return 'LOW'

tiers = [assign_tier(s) for s in scores]

# ── Intervention mapping ──────────────────────────────────────────────────────
INTERVENTIONS = {
    'HIGH':   'Nurse call within 24hrs | PCP appt within 7 days | Care coordination referral | Daily SMS monitoring',
    'MEDIUM': '48-hr follow-up call | PCP appt within 14 days | Pharmacist med reconciliation',
    'LOW':    'Standard discharge instructions | Patient portal message at day 7',
}

# ── Build output DataFrame ────────────────────────────────────────────────────
risk_output = model_df[['stay_id', 'subject_id', 'outtime',
                         'readmit_30d', 'readmit_60d', 'readmit_90d']].copy().reset_index(drop=True)
risk_output['risk_score']      = scores
risk_output['risk_tier']       = tiers
risk_output['readmit_prob_30d']= probs.round(3)
risk_output['interventions']   = [INTERVENTIONS[t] for t in tiers]

# ── Show distribution ─────────────────────────────────────────────────────────
tier_counts = risk_output['risk_tier'].value_counts()
print('=== Risk Tier Distribution ===')
for tier in ['HIGH', 'MEDIUM', 'LOW']:
    n = tier_counts.get(tier, 0)
    pct = n / len(risk_output) * 100
    print(f'  {tier:8s}: {n:3d} patients ({pct:.1f}%)')

print(f'\nMean risk score: {scores.mean():.1f}')
print(f'Max risk score:  {scores.max():.1f}')
print(f'Min risk score:  {scores.min():.1f}')

In [ ]:
# ── Risk score distribution — improved ──────────────────────────────────────
tier_colors = {'LOW': '#1D9E75', 'MEDIUM': '#BA7517', 'HIGH': '#E24B4A'}
tier_counts = risk_output['risk_tier'].value_counts()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.axvspan(0,  40, alpha=0.06, color='#1D9E75')
ax1.axvspan(40, 70, alpha=0.06, color='#BA7517')
ax1.axvspan(70, 100,alpha=0.06, color='#E24B4A')
for tier, color in tier_colors.items():
    mask = risk_output['risk_tier'] == tier
    ax1.hist(risk_output.loc[mask,'risk_score'], bins=20,
             color=color, alpha=0.85, label=f'{tier} (n={mask.sum()})')
ax1.axvline(40, color='#BA7517', linestyle='--', lw=1.5)
ax1.axvline(70, color='#E24B4A', linestyle='--', lw=1.5)
ax1.set_xlabel('Risk Score (0–100)', fontsize=12)
ax1.set_ylabel('Number of Patients', fontsize=12)
ax1.set_title('Risk Score Distribution by Tier', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.2)

tier_order = ['LOW', 'MEDIUM', 'HIGH']
counts_pie = [tier_counts.get(t,0) for t in tier_order]
colors_pie = [tier_colors[t] for t in tier_order]
wedges, texts, autotexts = ax2.pie(
    counts_pie, labels=tier_order, colors=colors_pie,
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    wedgeprops={'linewidth':2,'edgecolor':'white','width':0.6}
)
for at in autotexts: at.set_fontsize(12); at.set_fontweight('bold')
for t in texts: t.set_fontsize(12)
ax2.text(0, 0, f'{len(risk_output)}\npatients', ha='center', va='center',
         fontsize=13, fontweight='bold')
ax2.set_title('Patient Risk Tier Breakdown', fontsize=13, fontweight='bold')
plt.suptitle('IDRIB Risk Scoring — MIMIC-IV Demo Cohort', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/05_risk_score_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved.')


---
## SECTION 8 — SHAP Explainability
What features drove each patient's risk score? SHAP tells us.

In [ ]:
# ── Fixed SHAP computation — handles all 4 model types ───────────────────────
print('Computing SHAP values...')

if hasattr(best_model, 'named_steps'):
    clf   = best_model.named_steps['clf']
    X_shap = pd.DataFrame(
        best_model.named_steps['scaler'].transform(X),
        columns=FEATURE_COLS
    )
else:
    clf    = best_model
    X_shap = X.copy()

# All tree-based models use TreeExplainer — Logistic Regression uses LinearExplainer
from sklearn.linear_model import LogisticRegression as LR
if isinstance(clf, LR):
    explainer  = shap.LinearExplainer(clf, X_shap)
else:
    explainer  = shap.TreeExplainer(clf)   # works for RF, XGB, GradientBoosting

shap_values = explainer.shap_values(X_shap)

# Normalize to 2D (some models return 3D or list)
if isinstance(shap_values, list):
    shap_values = shap_values[1]
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

print(f'✅ SHAP values computed. Shape: {shap_values.shape}')

# Beeswarm plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, plot_type='dot',
                  feature_names=FEATURE_COLS, show=False, max_display=15)
plt.title('SHAP Feature Importance — Top 15 Risk Drivers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/06_shap_beeswarm.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ SHAP beeswarm saved')

In [ ]:
# Fix for SHAP values shape issue
# For Random Forest / XGBoost returning (n_samples, n_classes, n_features) or 2D array

if isinstance(shap_values, np.ndarray):
    if shap_values.ndim == 3:
        # 3D array: take class 1 (positive class)
        shap_vals_2d = shap_values[:, :, 1] if shap_values.shape[2] == 2 else shap_values[:, 1, :]
    elif shap_values.ndim == 2:
        shap_vals_2d = shap_values
    else:
        shap_vals_2d = shap_values
elif isinstance(shap_values, list):
    shap_vals_2d = np.array(shap_values[1])
else:
    shap_vals_2d = np.array(shap_values)

print(f"SHAP values shape fixed: {shap_vals_2d.shape}")
print(f"Features: {len(FEATURE_COLS)}")

# Global feature importance bar chart
mean_abs_shap = np.abs(shap_vals_2d).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=FEATURE_COLS).sort_values(ascending=True)
top15 = shap_importance.tail(15)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top15.index, top15.values, color='#378ADD', alpha=0.85, height=0.6)
ax.set_xlabel('Mean |SHAP Value| (Impact on Risk Score)', fontsize=12)
ax.set_title('Top 15 Features Driving ICU Readmission Risk', fontsize=13, fontweight='bold')
for bar, val in zip(bars, top15.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/07_shap_importance_bar.png', bbox_inches='tight')
plt.show()
print('✅ SHAP bar chart saved')

# Add top 3 drivers to each patient
shap_df = pd.DataFrame(shap_vals_2d, columns=FEATURE_COLS)
def top3_drivers(row_shap):
    top = row_shap.abs().nlargest(3).index.tolist()
    return ' | '.join([f.replace('_', ' ') for f in top])
risk_output['top_risk_drivers'] = [top3_drivers(shap_df.iloc[i]) for i in range(len(shap_df))]
print('✅ Top drivers added to each patient')
print(risk_output[['subject_id','risk_score','risk_tier','top_risk_drivers']].head(5))

---
## SECTION 9 — Export
Save the final risk score table and print a sample patient scorecard.

In [ ]:
# ── Save final risk score CSV ─────────────────────────────────────────────────
risk_output.to_csv('/content/drive/MyDrive/IDRIB_project/outputs/patient_risk_scores.csv', index=False)
print('✅ patient_risk_scores.csv saved → /content/drive/MyDrive/IDRIB_project/outputs/')
print(f'   Columns: {list(risk_output.columns)}')
print(f'   Rows: {len(risk_output)}')

In [ ]:
# ── Sample Patient Scorecard ──────────────────────────────────────────────────
# Show a printable summary for a single HIGH-risk patient
high_risk = risk_output[risk_output['risk_tier'] == 'HIGH']
if len(high_risk) > 0:
    sample = high_risk.sort_values('risk_score', ascending=False).iloc[0]
else:
    sample = risk_output.sort_values('risk_score', ascending=False).iloc[0]

print('=' * 55)
print('   IDRIB PATIENT DISCHARGE RISK SCORECARD')
print('=' * 55)
print(f'  Subject ID      : {int(sample["subject_id"])}')
print(f'  ICU Stay ID     : {int(sample["stay_id"])}')
print(f'  Discharge Time  : {sample["outtime"]}')
print(f'  Risk Score      : {sample["risk_score"]} / 100')
print(f'  Risk Tier       : ⚠️  {sample["risk_tier"]}')
print(f'  30-Day Readmit  : {"YES" if sample["readmit_30d"]==1 else "NO"} (actual)')
print(f'  Readmit Prob    : {sample["readmit_prob_30d"]:.1%}')
print(f'  Top Drivers     : {sample["top_risk_drivers"]}')
print()
print('  RECOMMENDED INTERVENTIONS:')
for action in sample['interventions'].split(' | '):
    print(f'    → {action}')
print('=' * 55)

---
## SECTION 10 — Clinical Evaluation
Confusion matrix, sensitivity/specificity, and readmission rate by tier.

In [ ]:
# ── Clinical metrics: confusion matrix + sensitivity/specificity ──────────────
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import confusion_matrix
import matplotlib.gridspec as gridspec

y_prob_best = best_model.predict_proba(X)[:, 1]
y_pred = (y_prob_best >= 0.35).astype(int)
cm = confusion_matrix(y, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp+fp)>0 else 0

fig = plt.figure(figsize=(16, 5))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# Confusion matrix
ax1 = fig.add_subplot(gs[0])
ax1.imshow(cm, cmap='Blues')
ax1.set_xticks([0,1]); ax1.set_yticks([0,1])
ax1.set_xticklabels(['Predicted\nNo Readmit','Predicted\nReadmit'], fontsize=10)
ax1.set_yticklabels(['Actual\nNo Readmit','Actual\nReadmit'], fontsize=10)
for ii in range(2):
    for jj in range(2):
        ax1.text(jj, ii, str(cm[ii,jj]), ha='center', va='center',
                 fontsize=18, fontweight='bold',
                 color='white' if cm[ii,jj]>cm.max()/2 else '#2C2C2A')
ax1.set_title('Confusion Matrix\n(threshold=0.35)', fontsize=12, fontweight='bold')

# Clinical metrics bar
ax2 = fig.add_subplot(gs[1])
metrics = {'Sensitivity\n(Recall)': sensitivity, 'Specificity': specificity, 'Precision\n(PPV)': ppv}
bars2 = ax2.bar(metrics.keys(), metrics.values(),
                color=['#E24B4A','#1D9E75','#378ADD'], alpha=0.85, width=0.5)
ax2.axhline(0.75, color='gray', linestyle='--', lw=1.5, label='Target=0.75')
ax2.set_ylim(0, 1.15); ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Clinical Performance Metrics', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
for bar2, val in zip(bars2, metrics.values()):
    ax2.text(bar2.get_x()+bar2.get_width()/2, val+0.02,
             f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')

# Precision-recall curve
ax3 = fig.add_subplot(gs[2])
prec, rec, _ = precision_recall_curve(y, y_prob_best)
ap = average_precision_score(y, y_prob_best)
ax3.plot(rec, prec, color='#7F77DD', lw=2.5, label=f'AP={ap:.3f}')
ax3.fill_between(rec, prec, alpha=0.1, color='#7F77DD')
ax3.axhline(y.mean(), color='gray', linestyle='--', lw=1.5, label=f'Baseline={y.mean():.2f}')
ax3.set_xlabel('Recall', fontsize=12); ax3.set_ylabel('Precision', fontsize=12)
ax3.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10); ax3.set_xlim([0,1]); ax3.set_ylim([0,1.05]); ax3.grid(alpha=0.2)

plt.suptitle('IDRIB Clinical Evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/08_clinical_metrics.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Sensitivity : {sensitivity:.1%}  — readmissions caught')
print(f'Specificity : {specificity:.1%}  — non-readmissions correctly cleared')
print(f'Precision   : {ppv:.1%}  — when flagged HIGH, this often correct')
print(f'True positives (caught): {tp}  |  False negatives (missed): {fn}')
print('Chart saved.')


In [ ]:
# ── Readmission rate by risk tier — clinical validation ───────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
tier_order2 = ['LOW', 'MEDIUM', 'HIGH']
tier_colors2 = ['#1D9E75', '#BA7517', '#E24B4A']
readmit_by_tier = risk_output.groupby('risk_tier')['readmit_30d'].agg(['mean','sum','count'])
readmit_by_tier.columns = ['rate','n_readmit','n_total']
rates2 = [readmit_by_tier.loc[t,'rate'] if t in readmit_by_tier.index else 0 for t in tier_order2]
ns2    = [readmit_by_tier.loc[t,'n_total'] if t in readmit_by_tier.index else 0 for t in tier_order2]

bars3 = ax1.bar(tier_order2, [r*100 for r in rates2],
                color=tier_colors2, alpha=0.85, width=0.5)
ax1.axhline(y.mean()*100, color='gray', linestyle='--', lw=2, label=f'Overall: {y.mean():.1%}')
ax1.set_ylabel('Actual 30-Day Readmission Rate (%)', fontsize=12)
ax1.set_title('Readmission Rate by Risk Tier', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10); ax1.set_ylim(0, 100)
for bar3, rate2, n2 in zip(bars3, rates2, ns2):
    ax1.text(bar3.get_x()+bar3.get_width()/2, bar3.get_height()+1,
             f'{rate2:.1%}\n(n={int(n2)})', ha='center', fontsize=11, fontweight='bold')

readmit_yes = risk_output[risk_output['readmit_30d']==1]['risk_score']
readmit_no  = risk_output[risk_output['readmit_30d']==0]['risk_score']
ax2.hist(readmit_no,  bins=15, alpha=0.6, color='#1D9E75', label=f'No readmit (n={len(readmit_no)})')
ax2.hist(readmit_yes, bins=15, alpha=0.75, color='#E24B4A', label=f'Readmitted (n={len(readmit_yes)})')
ax2.set_xlabel('Risk Score (0–100)', fontsize=12)
ax2.set_ylabel('Number of Patients', fontsize=12)
ax2.set_title('Risk Score: Readmitted vs Not Readmitted', fontsize=12, fontweight='bold')
ax2.legend(fontsize=11); ax2.grid(alpha=0.2)

plt.suptitle('Clinical Validation — Higher Risk Scores = More Readmissions?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/09_readmission_by_tier.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved.')


---
## SECTION 11 — Hosmer-Lemeshow Calibration Test
Tests whether predicted probabilities match observed readmission rates. A well-calibrated model is essential for clinical trust.

In [ ]:
# ── Hosmer-Lemeshow Calibration Test ────────────────────────────────────────
# H-L tests if predicted probabilities match observed rates.
# p > 0.05 = well calibrated (predicted risk matches reality)
# p < 0.05 = poorly calibrated (model over/under-estimates risk)
from scipy import stats
import numpy as np

def hosmer_lemeshow_test(y_true, y_prob, n_bins=10):
    """Hosmer-Lemeshow goodness-of-fit test."""
    df_hl = pd.DataFrame({'y':y_true, 'prob':y_prob})
    df_hl['bin'] = pd.qcut(df_hl['prob'], q=n_bins, duplicates='drop')
    grouped = df_hl.groupby('bin', observed=True)
    observed_pos  = grouped['y'].sum()
    observed_neg  = grouped['y'].count() - observed_pos
    expected_pos  = grouped['prob'].sum()
    expected_neg  = grouped['prob'].count() - expected_pos
    chi2 = (((observed_pos - expected_pos)**2 / expected_pos.clip(lower=1e-10)) +
            ((observed_neg - expected_neg)**2 / expected_neg.clip(lower=1e-10))).sum()
    df_free = len(grouped) - 2
    p_value = 1 - stats.chi2.cdf(chi2, df_free)
    return chi2, p_value, df_free, grouped

y_prob_hl = best_model.predict_proba(X)[:, 1]
chi2_stat, p_val, df_free, grouped = hosmer_lemeshow_test(y, y_prob_hl)

print('=== Hosmer-Lemeshow Calibration Test ===')
print(f'  Chi-square statistic : {chi2_stat:.3f}')
print(f'  Degrees of freedom   : {df_free}')
print(f'  p-value              : {p_val:.4f}')
print()
if p_val > 0.05:
    print('  ✅ WELL CALIBRATED (p > 0.05)')
    print('     Predicted probabilities match observed readmission rates.')
    print('     Model can be trusted for clinical probability estimates.')
else:
    print('  ⚠️  POORLY CALIBRATED (p < 0.05)')
    print('     Predicted probabilities do not perfectly match observed rates.')
    print('     This is expected on 100 patients — will improve with full dataset.')

# Calibration plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# H-L decile plot
obs  = grouped['y'].mean().values
pred = grouped['prob'].mean().values
ax1.plot([0,1],[0,1],'k--',lw=1.5,label='Perfect calibration')
ax1.scatter(pred, obs, s=120, color='#378ADD', zorder=5, label='Decile groups')
for p_, o_ in zip(pred, obs):
    ax1.plot([p_,p_],[p_,o_], color='#E24B4A', lw=1.5, alpha=0.6)
ax1.set_xlabel('Mean Predicted Probability', fontsize=12)
ax1.set_ylabel('Observed Readmission Rate',  fontsize=12)
ax1.set_title(f'Hosmer-Lemeshow Calibration Plot\n'
              f'Chi²={chi2_stat:.2f}, p={p_val:.3f} '
              f'({"✅ Calibrated" if p_val>0.05 else "⚠️ Review needed"})',
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_xlim([-0.05,1.05]); ax1.set_ylim([-0.05,1.05])
ax1.grid(alpha=0.2)

# Probability histogram
ax2.hist(y_prob_hl[y==0], bins=20, alpha=0.6, color='#1D9E75', label='No readmit')
ax2.hist(y_prob_hl[y==1], bins=20, alpha=0.75, color='#E24B4A', label='Readmitted')
ax2.set_xlabel('Predicted Readmission Probability', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Predicted Probability Distribution\nby Actual Outcome', fontsize=12, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/10_hosmer_lemeshow.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Chart saved.')


---
## SECTION 12 — Multi-Window Validation: 60-Day & 90-Day
Retrains and evaluates the best model on 60-day and 90-day readmission targets. Required KPI from the professor's proposal.

In [ ]:
# ── Validate on 60-day and 90-day readmission targets ───────────────────────
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_auc_score, average_precision_score

windows = {
    '30-day': 'readmit_30d',
    '60-day': 'readmit_60d',
    '90-day': 'readmit_90d',
}

window_results = {}
print('=== Multi-Window Readmission Validation ===')
print(f'{"Window":10s}  {"N Readmit":10s}  {"Rate":8s}  {"ROC-AUC":10s}  {"PR-AUC":10s}')
print('-' * 55)

for window_name, target_col in windows.items():
    y_w = model_df[target_col].values
    n_pos = y_w.sum()
    rate  = y_w.mean()

    # Build pipeline for this window
    k_nn = min(4, n_pos - 1) if n_pos > 1 else 1
    pipe = ImbPipeline([
        ('smote', SMOTE(random_state=42, k_neighbors=k_nn)),
        ('clf',   GradientBoostingClassifier(
            n_estimators=100, max_depth=2, learning_rate=0.05,
            random_state=42, subsample=0.8, min_samples_leaf=5))
    ])

    cv_w = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    try:
        roc_w = cross_val_score(pipe, X, y_w, cv=cv_w, scoring='roc_auc',           n_jobs=-1)
        pr_w  = cross_val_score(pipe, X, y_w, cv=cv_w, scoring='average_precision',  n_jobs=-1)
        window_results[window_name] = {
            'n_readmit': int(n_pos), 'rate': rate,
            'roc_mean': roc_w.mean(), 'roc_std': roc_w.std(),
            'pr_mean':  pr_w.mean(),  'pr_std':  pr_w.std()
        }
        print(f'{window_name:10s}  {int(n_pos):10d}  {rate:7.1%}  '
              f'{roc_w.mean():.3f}±{roc_w.std():.3f}  '
              f'{pr_w.mean():.3f}±{pr_w.std():.3f}')
    except Exception as e:
        print(f'{window_name:10s}  error: {e}')

# Comparison chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
wins = list(window_results.keys())
roc_means = [window_results[w]['roc_mean'] for w in wins]
roc_stds  = [window_results[w]['roc_std']  for w in wins]
pr_means  = [window_results[w]['pr_mean']  for w in wins]
pr_stds   = [window_results[w]['pr_std']   for w in wins]

colors_w = ['#378ADD','#1D9E75','#BA7517']
x_pos = range(len(wins))

bars1 = ax1.bar(x_pos, roc_means, yerr=roc_stds, color=colors_w, alpha=0.85,
                width=0.5, capsize=6, error_kw={'lw':2})
ax1.axhline(0.8, color='#E24B4A', linestyle='--', lw=1.5, label='Target=0.80')
ax1.axhline(0.5, color='gray',    linestyle=':',  lw=1,   label='Random=0.50')
ax1.set_xticks(x_pos); ax1.set_xticklabels(wins, fontsize=12)
ax1.set_ylim(0.3, 1.05)
ax1.set_ylabel('ROC-AUC', fontsize=12)
ax1.set_title('ROC-AUC by Readmission Window\n(Honest CV — SMOTE inside folds)',
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=10); ax1.grid(axis='y', alpha=0.3)
for bar, val, std in zip(bars1, roc_means, roc_stds):
    ax1.text(bar.get_x()+bar.get_width()/2, val+std+0.01,
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

bars2 = ax2.bar(x_pos, pr_means, yerr=pr_stds, color=colors_w, alpha=0.85,
                width=0.5, capsize=6, error_kw={'lw':2})
baselines = [window_results[w]['rate'] for w in wins]
for bl, xp, wn in zip(baselines, x_pos, wins):
    ax2.axhline(bl, color=colors_w[list(wins).index(wn)],
                linestyle=':', lw=1.5, alpha=0.7)
ax2.set_xticks(x_pos); ax2.set_xticklabels(wins, fontsize=12)
ax2.set_ylim(0, min(1.0, max(pr_means)+max(pr_stds)+0.2))
ax2.set_ylabel('PR-AUC (Average Precision)', fontsize=12)
ax2.set_title('PR-AUC by Readmission Window\n(Dotted lines = random baseline per window)',
              fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for bar, val, std in zip(bars2, pr_means, pr_stds):
    ax2.text(bar.get_x()+bar.get_width()/2, val+std+0.01,
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('IDRIB — 30 / 60 / 90-Day Readmission Model Performance',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/IDRIB_project/outputs/charts/11_multiwindow_validation.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved.')


In [ ]:
# ── Final summary — v4 complete ─────────────────────────────────────────────
print('\n' + '='*60)
print('  IDRIB PIPELINE v4 — RUN COMPLETE')
print('='*60)
print(f'  Model            : {best_name}')
print(f'  ROC-AUC (30-day) : {results_roc[best_name].mean():.3f} ± {results_roc[best_name].std():.3f}')
print(f'  PR-AUC  (30-day) : {results_pr[best_name].mean():.3f} ± {results_pr[best_name].std():.3f}')
print(f'  Baseline PR-AUC  : {y.mean():.3f} (random classifier)')
print(f'  H-L p-value      : {p_val:.4f} ({"Calibrated" if p_val>0.05 else "Review needed"})')
print(f'  Patients scored  : {len(risk_output)}')
print(f'  HIGH risk        : {(risk_output["risk_tier"]=="HIGH").sum()}')
print(f'  MEDIUM risk      : {(risk_output["risk_tier"]=="MEDIUM").sum()}')
print(f'  LOW risk         : {(risk_output["risk_tier"]=="LOW").sum()}')
print()
print('  MULTI-WINDOW RESULTS:')
for w, r in window_results.items():
    print(f'    {w}: ROC={r["roc_mean"]:.3f} | PR={r["pr_mean"]:.3f} | '
          f'N readmit={r["n_readmit"]} ({r["rate"]:.1%})')
print()
print('  NEW FEATURES ADDED:')
print('    Elixhauser comorbidity score (validated clinical index)')
print('    Ventilation flag (mechanical ventilation during ICU stay)')
print('    Vasopressor flag (vasopressor use during ICU stay)')
print('    Time-series trends (slope of vitals over full stay)')
print('    SMOTE inside CV folds (no data leakage)')
print()
print('  OUTPUT FILES:')
print('    /content/drive/MyDrive/IDRIB_project/outputs/patient_risk_scores.csv')
print('    /content/drive/MyDrive/IDRIB_project/models/risk_model.pkl')
print('    outputs/charts/ (11 charts)')
print('='*60)
